In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# Load datasets

trades = pd.read_csv("../data/historical_data.csv")
sentiment = pd.read_csv("../data/fear_greed_index.csv")

In [ ]:
print("Trades shape:", trades.shape)
print("Sentiment shape:", sentiment.shape)

In [ ]:
print("Trades columns:")
print(trades.columns)

print("\nSentiment columns:")
print(sentiment.columns)

In [ ]:
trades.head()

In [ ]:
sentiment.head()

In [ ]:
# Convert trader timestamp to datetime
trades["Timestamp IST"] = pd.to_datetime(
    trades["Timestamp IST"],
    dayfirst=True
)

# Create date-only column
trades["date"] = trades["Timestamp IST"].dt.date

# Convert sentiment date column
sentiment["date"] = pd.to_datetime(
    sentiment["date"]
).dt.date

In [ ]:
print(trades[["Timestamp IST", "date"]].head())

print("\n")

print(sentiment[["date", "classification"]].head())

In [ ]:
df = trades.merge(
    sentiment[["date", "classification", "value"]],
    on="date",
    how="left"
)

In [ ]:
print(df[[
    "date",
    "Coin",
    "Closed PnL",
    "classification",
    "value"
]].head())

In [ ]:
pnl_summary = df.groupby(
    "classification"
)["Closed PnL"].agg(
    ["mean", "sum", "count"]
)

print(pnl_summary)

In [ ]:
# Plot average pnl by sentiment

pnl_summary["mean"].plot(
    kind="bar",
    figsize=(8,5)
)

plt.title("Average Closed PnL by Market Sentiment")
plt.ylabel("Average Closed PnL")
plt.xlabel("Market Sentiment")
plt.xticks(rotation=45)

plt.show()

In [ ]:
df["is_profit"] = df["Closed PnL"] > 0

In [ ]:
# Win rate %

win_rate = df.groupby(
    "classification"
)["is_profit"].mean() * 100

print(win_rate)

In [ ]:
win_rate.plot(
    kind="bar",
    figsize=(8,5)
)

plt.title("Win Rate by Market Sentiment")
plt.ylabel("Winning Trades (%)")
plt.xlabel("Market Sentiment")
plt.xticks(rotation=45)

plt.show()

In [ ]:
trade_count = df.groupby(
    "classification"
).size()

print(trade_count)

In [ ]:
trade_count.plot(
    kind="bar",
    figsize=(8,5)
)

plt.title("Number of Trades by Market Sentiment")
plt.ylabel("Trade Count")
plt.xlabel("Market Sentiment")
plt.xticks(rotation=45)

plt.show()

In [ ]:
side_summary = df.groupby(
    ["classification", "Side"]
)["Closed PnL"].mean()

print(side_summary)

In [ ]:
side_plot = df.pivot_table(
    values="Closed PnL",
    index="classification",
    columns="Side",
    aggfunc="mean"
)

print(side_plot)

In [ ]:
side_plot.plot(
    kind="bar",
    figsize=(10,5)
)

plt.title("Average PnL by Market Sentiment and Trade Side")
plt.ylabel("Average Closed PnL")
plt.xlabel("Market Sentiment")
plt.xticks(rotation=45)

plt.show()

In [ ]:
top_traders = df.groupby("Account").agg(
    total_pnl=("Closed PnL", "sum"),
    avg_pnl=("Closed PnL", "mean"),
    trades=("Account", "count")
)

In [ ]:
top_10 = top_traders.sort_values(
    "total_pnl",
    ascending=False
).head(10)

print(top_10)

In [ ]:
# Aggregate trader behavior per day

daily_features = df.groupby("date").agg(
    avg_pnl=("Closed PnL", "mean"),
    total_pnl=("Closed PnL", "sum"),
    trade_count=("Account", "count"),
    avg_fee=("Fee", "mean"),
    buy_count=("Side", lambda x: (x == "BUY").sum()),
    sell_count=("Side", lambda x: (x == "SELL").sum()),
    classification=("classification", "first")
).reset_index()
daily_features = df.groupby("date").agg(
    avg_pnl=("Closed PnL", "mean"),
    total_pnl=("Closed PnL", "sum"),
    trade_count=("Account", "count"),
    avg_fee=("Fee", "mean"),
    buy_count=("Side", lambda x: (x == "BUY").sum()),
    sell_count=("Side", lambda x: (x == "SELL").sum()),
    classification=("classification", "first")
).reset_index()

In [ ]:
daily_features.head()

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# Features
X = daily_features[
    [
        "avg_pnl",
        "total_pnl",
        "trade_count",
        "avg_fee",
        "buy_count",
        "sell_count"
    ]
]

# Target
y = daily_features["classification"]

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# Train
model = RandomForestClassifier(
    random_state=42
)

model.fit(X_train, y_train)

# Predict
y_pred = model.predict(X_test)

# Accuracy
print("Accuracy:", accuracy_score(y_test, y_pred))

# Full report
print(classification_report(y_test, y_pred))

In [ ]:
# Remove rows with missing values

daily_features = daily_features.dropna()